<a href="https://colab.research.google.com/github/guillermojleonr/skill-test-alca-data/blob/main/PruebaTecnica_Guillermo_Leon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
# @title
# ==========================================
# CONFIGURACIÓN DEL ENTORNO (EJECUTAR PRIMERO)
# ==========================================
import pandas as pd
import sqlite3

# 1) URLs de archivos crudos (Raw) en GitHub
urls = {
    'clientes': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/clientes.csv",
    'ventas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/ventas.csv",
    'trabajadores': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/trabajadores.csv",
    'cargos': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/cargos.csv",
    'areas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/areas.csv"
}

print("Preparando base de datos 'techstore.db'.. .")

# 2) Crear la conexión y la base de datos
conn = sqlite3.connect('techstore.db')

# 3) Leer cada CSV e insertarlo como tabla en la base de datos
for tabla, url in urls.items():
    try:
        df = pd.read_csv(url, sep=None, engine='python')
        df.to_sql(tabla, conn, index=False, if_exists='replace')
        print(f"Tabla '{tabla}' cargada exitosamente.")
    except Exception as e:
        print(f"Error al cargar la tabla '{tabla}': {e}")

conn.close()
print("¡Entorno listo! La base de datos relacional está disponible para ser consultada.")

Preparando base de datos 'techstore.db'.. .
Tabla 'clientes' cargada exitosamente.
Tabla 'ventas' cargada exitosamente.
Tabla 'trabajadores' cargada exitosamente.
Tabla 'cargos' cargada exitosamente.
Tabla 'areas' cargada exitosamente.
¡Entorno listo! La base de datos relacional está disponible para ser consultada.


**Contexto de la Prueba: E-Commerce "TechStore"**

*Escenario: Eres el nuevo analista de datos de TechStore. El equipo ha notado una fluctuación en los ingresos durante el último trimestre y necesita entender qué está pasando.*

**Instrucciones Iniciales:**

1.   La prueba tiene una duración estimada de 45 mins, pero tendrás 72 horas para enviar tus resultados.
2.	Luego de terminar tus ejercicios envíame el enlace en compartir para tu entorno donde fue desarrollado.
3.	Cualquier archivo externo generado debes subirlo y compartir el enlace en una nueva hoja de google colab, en caso de usar hojas de cálculo de Google debes copiar el enlace en otra hoja de google colab, recuerda titular dicho enlace para saber que iré a mirar.
4.	Dentro de la prueba, ve a Archivo > Guardar una copia en Drive para tener tu propia versión editable de esta prueba.
5.	Ejecuta la primera celda de este cuaderno. Esto generará automáticamente una base de datos SQLite llamada techstore.db en este entorno.
6.	El diccionario de datos disponible en la base de datos es:
*   clientes: id, rut, nombre, segundo_nombre, apellido, segundo_apellido, edad, fecha_nacimiento
*   ventas: id, id_cliente, id_vendedor, dte, total_dte, estado, descripcion
*   trabajadores: id, id_cargo, nombre, apellido, rut
*   cargos: id, nombre_cargo, descripcion, id_area
*   areas: id, nombre_area, descripcion

*disclaimer: Puedes usar IA, sin embargo, revisaré cada linea de código para validar la efectividad y también la totalidad del cumplimiento de las respuestas.*

**Parte 1: Conexión y Extracción (SQL & Python)**

*Conéctate a la base de datos techstore.db usando Python (por ejemplo, con la librería sqlite3 o sqlalchemy). Luego, resuelve las siguientes consultas ejecutando SQL puro dentro de tu código Python:*
1.   **Ventas Totales:** Escribe una consulta para calcular el ingreso total generado por órdenes con estado "Entregado" y el % del total de ventas que representa.
2.   **Top Clientes:** Encuentra los 5 usuarios que han gastado más dinero históricamente, mostrando su id, rut y el total_dte gastado.
3.   **Retención:** Calcula el porcentaje de usuarios que realizaron una segunda compra dentro de los 200 dte siguientes (si alguien compró el dte 11721 y compró nuevamente antes ó justamente en el dte 11921 es considerado).

In [76]:
# Set up
conn = sqlite3.connect('techstore.db')
cursor = conn.cursor()

#Solución

En la parte 1 como el enunciado dice ejecutando "SQL Puro" traté de usar la mayor cantidad de SQL que pude, evitando usar pandas. Por defecto prefiero SQL antes que pandas a menos que la consulta sea muy compleja

### Parte 1, Tarea 1: Ventas totales

In [81]:
# 1. Calcular el ingreso total generado por órdenes con estado "Entregado"
query_entregado = """
SELECT SUM(total_dte)
FROM ventas
WHERE estado = 'Entregado';
"""
cursor.execute(query_entregado)
total_entregado = cursor.fetchone()[0]

# 2. Calcular el total de todas las ventas
query_total_ventas = """
SELECT SUM(total_dte)
FROM ventas;
"""
cursor.execute(query_total_ventas)
total_ventas = cursor.fetchone()[0]

# 3. Calcular el porcentaje
porcentaje_entregado = (total_entregado / total_ventas) * 100 if total_ventas else 0

# 4. No entregado
total_no_entregado = total_ventas - total_entregado

print(f"Ingreso total generado por órdenes 'Entregado': {total_entregado:.2f}")
print(f"Porcentaje del total de ventas que representa: {porcentaje_entregado:.2f}%")
print(f"Ingreso total generado por órdenes 'No entregado': {total_no_entregado:.2f}")

Ingreso total generado por órdenes 'Entregado': 477191018.00
Porcentaje del total de ventas que representa: 81.06%
Ingreso total generado por órdenes 'No entregado': 111482466.00


### Parte 1, Tarea 2: Top Clientes por Gasto Histórico

In [65]:
query_top_clientes = """
SELECT
    c.id AS id_cliente,
    c.rut AS rut_cliente,
    c.nombre || ' ' || c.apellido AS nombre_completo_cliente,
    SUM(v.total_dte) AS total_gastado
FROM
    clientes c
JOIN
    ventas v ON c.id = v.id_cliente
GROUP BY
    c.id, c.rut, c.nombre, c.apellido
ORDER BY
    total_gastado DESC
LIMIT 5;
"""

cursor.execute(query_top_clientes)
top_clientes = cursor.fetchall()

# Mostrar los resultados
print("Top 5 clientes que han gastado más dinero históricamente:")
for cliente in top_clientes:
    print(f"ID: {cliente[0]}, RUT: {cliente[1]}, Nombre: {cliente[2]}, Total Gastado: {cliente[3]:.2f}")



Top 5 clientes que han gastado más dinero históricamente:
ID: 82, RUT: 21866669-8, Nombre: Sofía Sepúlveda, Total Gastado: 7125123.00
ID: 81, RUT: 10212267-4, Nombre: Natalia Valenzuela, Total Gastado: 5663000.00
ID: 94, RUT: 11686548-3, Nombre: Javier Donoso, Total Gastado: 5621201.00
ID: 145, RUT: 14605016-6, Nombre: Paulina Medina, Total Gastado: 5604037.00
ID: 48, RUT: 19301497-6, Nombre: Patricia Herrera, Total Gastado: 5106230.00


### Parte 1, Tarea 3: Retención de Clientes

3.   **Retención:** Calcula el porcentaje de usuarios que realizaron una segunda compra dentro de los 200 dte siguientes (si alguien compró el dte 11721 y compró nuevamente antes ó justamente en el dte 11921 es considerado).

In [66]:
# Cargar la tabla de ventas en un DataFrame
query = """
SELECT COUNT(q.id_cliente)
FROM (
	SELECT
		id_cliente,
		COUNT(dte) as cantidad_dte
	FROM ventas
	WHERE dte BETWEEN 11721 AND 11921
	GROUP BY id_cliente
	HAVING cantidad_dte > 1
	ORDER BY id_cliente, dte
) AS q;
"""
response = cursor.execute(query)
cantidad_clientes_con_segunda_compra = cursor.fetchall()
print(f"Cantidad de clientes más de una compra: {cantidad_clientes_con_segunda_compra}")

query = """

SELECT COUNT(q.id_cliente)
FROM (
	SELECT
		id_cliente,
		COUNT(dte) as cantidad_dte
	FROM ventas
	WHERE dte BETWEEN 11721 AND 11921
	GROUP BY id_cliente
	ORDER BY id_cliente, dte
) AS q;
"""
response = cursor.execute(query)
cantidad_clientes_totales = cursor.fetchall()
print(f"Cantidad de clientes totales: {cantidad_clientes_totales}")

porcentaje_clientes_con_segunda_compra = (cantidad_clientes_con_segunda_compra[0][0] / cantidad_clientes_totales[0][0]) * 100
print(f"Porcentaje de clientes con segunda compra: {porcentaje_clientes_con_segunda_compra:.2f}%")

Cantidad de clientes más de una compra: [(20,)]
Cantidad de clientes totales: [(82,)]
Porcentaje de clientes con segunda compra: 24.39%


## Parte 2: Análisis Exploratorio (Python)

*Trae las tablas necesarias a DataFrames de Pandas y resuelve:*

1.	**Limpieza:** Identifica y muestra los valores nulos o duplicados en la tabla de clientes. Comenta tu criterio.
2.	**Métricas:** Identifica el top 10 de total_dte más altos, el top 3 de clientes por volumen de transacciones (cantidad de compras), y el top 3 de vendedores (id_vendedor) con mayor monto acumulado, solo considera vendedores, si hay otros cargos comerciales que han realizado ventas debes ignorarlos.
3.	**Volumen:** Crea un DataFrame del top 10 de productos mas vendidos ordenado de mayor a menor que muestre el id de la venta, nombre del producto y unidades vendidas (deberás procesar la columna descripcion).


In [67]:

# Esta parte del enunciado habla de usar Pandas, entonces priorizo el uso de pandas sobre SQL

# Set Up
df_ventas = pd.read_sql_query("SELECT * FROM ventas", conn)
df_clientes = pd.read_sql_query("SELECT * FROM clientes", conn)
df_trabajadores = pd.read_sql_query("SELECT * FROM trabajadores", conn)
df_cargos = pd.read_sql_query("SELECT * FROM cargos", conn)
df_areas = pd.read_sql_query("SELECT * FROM areas", conn)


### Parte 2, tarea 1

In [68]:
"""
Limpieza: Identifica y muestra los valores nulos o duplicados en la tabla de clientes. Comenta tu criterio.
"""

df_clientes_nulos = df_clientes.isnull().sum()
print(f"Valores nulos por columna:\n{df_clientes_nulos}")
print("-----------------------------------------")

df_clientes_duplicados = df_clientes[df_clientes.duplicated(subset = ["rut"])]
print(f"Valores duplicados:\n{df_clientes_duplicados}")
print("-----------------------------------------")

#Resultados

"""
1. No hay valores nulos en la tabla clientes a excepción del campo edad, considero al cliente "válido"
con la salvedad que representa no tener la edad del cliente.

2. Hay dos clientes cuyo RUT está duplicado, registro debería ser eliminado y colocar un constraint de unicidad
en la tabla en el campo RUT para evitar este problema en el futuro y filtrar en el pipeline también.
"""

Valores nulos por columna:
id                  0
rut                 0
nombre              0
segundo_nombre      0
apellido            0
segundo_apellido    0
edad                4
fecha_nacimiento    0
dtype: int64
-----------------------------------------
Valores duplicados:
      id         rut   nombre segundo_nombre apellido segundo_apellido  edad  \
198  199  10352896-8  Javiera      Valentina    Silva            Rojas  36.0   
250  251  12626181-0    Diego         Manuel     Soto         Castillo  60.0   

    fecha_nacimiento  
198       01-05-1990  
250       25-09-1966  
-----------------------------------------


'\n1. No hay valores nulos en la tabla clientes a excepción del campo edad, considero al cliente "válido" \ncon la salvedad que representa no tener la edad del cliente.\n\n2. Hay dos clientes cuyo RUT está duplicado, registro debería ser eliminado y colocar un constraint de unicidad \nen la tabla en el campo RUT para evitar este problema en el futuro y filtrar en el pipeline también.\n'

### Parte 2, tarea 2

In [69]:
"""
Top 10 de total_dte más altos
"""
# Cargar la tabla de ventas en un DataFrame
df_ventas = pd.read_sql_query("SELECT * FROM ventas", conn)

# Identificar el top 10 de total_dte más altos
top_10_ventas = df_ventas.nlargest(10, 'total_dte')

print("Top 10 ventas (total_dte más altos):")
display(top_10_ventas[['id', 'total_dte', 'id_cliente', 'id_vendedor', 'estado', 'descripcion']])

Top 10 ventas (total_dte más altos):


,id,total_dte,id_cliente,id_vendedor,estado,descripcion
320,321,999825,75,12,Entregado,"1x Hub USB-C 8 en 1, 3x Licencia Software Anti..."
483,484,998087,40,2,Entregado,"1x Silla de Escritorio Ergonómica, 1x Escáner ..."
627,628,996482,45,2,Entregado,"1x Monitor LED 24' Full HD, 3x Silla de Escrit..."
661,662,996396,182,7,Entregado,"3x Disco Duro Externo 2TB, 1x Teclado Mecánico..."
458,459,996276,144,10,Sin entregar,"2x Teclado Mecánico RGB, 2x Escáner de Documen..."
114,115,995217,201,5,Entregado,"1x Disco Duro Externo 2TB, 2x Hub USB-C 8 en 1..."
734,735,995031,110,12,Entregado,"3x Memoria RAM 16GB DDR4, 1x Disco Duro Extern..."
442,443,994428,106,7,Entregado,1x Mouse Ergonómico Inalámbrico
178,179,994032,163,3,Entregado,"1x Hub USB-C 8 en 1, 1x Memoria RAM 16GB DDR4,..."
207,208,992339,72,15,Entregado,"3x Laptop Corporate 14', 3x Escáner de Documen..."


In [70]:
"""
Top 3 de clientes por volumen de transacciones (cantidad de compras)
"""

top = df_ventas['id_cliente'].value_counts().nlargest(30).reset_index()
top.columns = ['id_cliente', 'cantidad_compras']

# Unir con la tabla de clientes para obtener nombre y rut
top = top.merge(
    df_clientes[['id', 'rut', 'nombre', 'apellido']],
    left_on='id_cliente', right_on='id',
    how='left'
)

# Ranking con empates
top['posicion'] = (
    top['cantidad_compras']
    .rank(method='dense', ascending=False)
)

top = top[top['posicion'] <= 3]

display(top)

#Resultados

"""
Hay empate en el top 3. No desempato para no hacerlo arbitrariamente, podría desempatar por monto de venta, pero para mantener la instrucción lo dejo así
"""

,id_cliente,cantidad_compras,id,rut,nombre,apellido,posicion
0,147,11,147,20416825-3,Nicolás,Silva,1.0
1,81,10,81,10212267-4,Natalia,Valenzuela,2.0
2,82,10,82,21866669-8,Sofía,Sepúlveda,2.0
3,48,10,48,19301497-6,Patricia,Herrera,2.0
4,30,10,30,11548511-3,Isabel,Martínez,2.0
5,148,10,148,15183033-1,Francisca,Espinoza,2.0
6,84,10,84,15407373-6,Alejandra,Medina,2.0
7,73,10,73,15871360-8,Constanza,Torres,2.0
8,94,9,94,11686548-3,Javier,Donoso,3.0
9,49,9,49,20326626-K,Francisco,Vergara,3.0


'\nHay empate en el top 3. No desempato para no hacerlo arbitrariamente, podría desempatar por monto de venta, pero para mantener la instrucción lo dejo así\n'

In [71]:
"""
Top 3 de vendedores (id_vendedor) con mayor monto acumulado, solo considera vendedores, si hay otros cargos comerciales que han realizado ventas debes ignorarlos.
"""

vendedores = df_trabajadores[df_trabajadores['id_cargo'].isin([2, 3])] #2 y 3 son ejecutivos de ventas
ventas_vendedores = df_ventas.merge(vendedores, left_on='id_vendedor', right_on='id', how='left')
ventas_vendedores = ventas_vendedores[['id_vendedor', 'nombre', 'apellido', 'total_dte']]
top = ventas_vendedores.groupby(['id_vendedor', 'nombre', 'apellido']).sum().nlargest(3, 'total_dte')
display(top)



,,,total_dte
id_vendedor,nombre,apellido,
5,Felipe,Sepúlveda,40264872
13,Manuel,Cortés,38625479
7,Andrea,Martínez,36945022


### Parte 2, tarea 3

In [79]:
import re

"""
top 10 de productos mas vendidos ordenado de mayor a menor que muestre el id de la venta, nombre del producto y
unidades vendidas (deberás procesar la columna descripcion).
"""

# Productos se encuentran en la columna df_ventas['descripcion'] separados por comas

ventas_productos = df_ventas[['id', 'descripcion']].copy()


ventas_productos['descripcion'] = ventas_productos['descripcion'].str.rstrip(',') # Remover trailing commas en caso de haberlas
split_products = ventas_productos['descripcion'].str.split(',', expand=True) # dividir según ','

ventas_productos_expanded = pd.concat([ventas_productos['id'], split_products], axis=1) # concatenar

# Unpivot
product_cols = [col for col in ventas_productos_expanded.columns if isinstance(col, int)] # Columns 0, 1, 2, ... de producto
ventas_productos_melted = ventas_productos_expanded.melt(id_vars=['id'], value_vars=product_cols, value_name='producto_detalle')

# Filtrar None y empty ('')
ventas_productos_melted = ventas_productos_melted.dropna(subset=['producto_detalle'])
ventas_productos_melted['producto_detalle'] = ventas_productos_melted['producto_detalle'].str.strip()
ventas_productos_melted = ventas_productos_melted[ventas_productos_melted['producto_detalle'] != '']

# Función para parsear (e.g., '3x Laptop' -> quantity=3, name='Laptop')
def parse_product_detail(detail):
    match_objeto = re.match(r'(\d+)x\s+(.*)', detail)
    if match_objeto:
        return int(match_objeto.group(1)), match_objeto.group(2).strip() # Group1 devuelve la cantidad vendida y Group2 la descripción, stripeada
    else:
        # Asumimos cantidad 1 si no hay 1x, 2x, 3x...
        return 1, detail.strip()

# Aplicando parseo
ventas_productos_melted[['unidades_vendidas', 'nombre_producto']] = \
    ventas_productos_melted['producto_detalle'].apply(lambda x: pd.Series(parse_product_detail(x)))

# Agregamos

## ADVERTENCIA. El enunciado dice: "que muestre el id de la venta" La única forma que se me ocurre para mostrar el id de la venta en una vista agregada
## por producto es concatenando el id de la venta en un campo, aunque la vista no tenga mucho sentido
top_productos_agregados = (
    ventas_productos_melted
    .groupby('nombre_producto')
    .agg(
        unidades_vendidas=('unidades_vendidas', 'sum'),
        ids_venta=('id', lambda x: ', '.join(map(str, sorted(x.unique()))))
    )
    .reset_index()
)

print("----------------------------")
print("Concatenando el id de la venta en un campo")
display(top_productos_agregados.sort_values(by='unidades_vendidas', ascending=False).head(10))

## Otra forma es agrupando por nombre_producto y dejando visible el campo id (id de la venta) solo como un valor representativo

top_productos_agregados = (
    ventas_productos_melted
    .groupby('nombre_producto')
    .agg(
        unidades_vendidas=('unidades_vendidas', 'sum'),
        ids_venta=('id', 'first')
    )
    .reset_index()
)

print("----------------------------")
print("Dejando visible el campo id sólo como un campo representativo")
display(top_productos_agregados.sort_values(by='unidades_vendidas', ascending=False).head(10))

## Otra forma sería simular una "window function" (concepto SQL), para mostrar una tabla de detalle con cada venta y su id correspondiente
## sin colapsar con un group_by pero con una columna adicional "unidades vendidas totales" que mostraría la suma total de ese producto
## Pero en este caso si hacemos un top 10 la vista de detalle carece de sentido, dejo el código sólo como referencia



----------------------------
Concatenando el id de la venta en un campo


,nombre_producto,unidades_vendidas,ids_venta
2,Escáner de Documentos Portátil,551,"1, 2, 3, 7, 8, 15, 16, 17, 23, 26, 27, 31, 35,..."
9,Mouse Ergonómico Inalámbrico,532,"1, 2, 3, 5, 6, 13, 16, 17, 20, 21, 25, 27, 28,..."
5,Laptop Corporate 14',504,"4, 7, 9, 14, 32, 33, 55, 57, 60, 63, 70, 72, 7..."
3,Hub USB-C 8 en 1,502,"11, 19, 33, 35, 37, 39, 47, 50, 63, 66, 71, 75..."
7,Memoria RAM 16GB DDR4,492,"3, 7, 16, 23, 33, 34, 44, 49, 54, 56, 64, 73, ..."
1,Disco Duro Externo 2TB,492,"1, 19, 26, 27, 29, 31, 37, 38, 40, 46, 48, 49,..."
6,Licencia Software Antivirus 1 Año,486,"4, 12, 13, 20, 21, 24, 30, 35, 39, 40, 41, 42,..."
10,Silla de Escritorio Ergonómica,479,"1, 3, 8, 11, 14, 24, 30, 43, 46, 48, 50, 58, 6..."
11,Teclado Mecánico RGB,475,"11, 19, 36, 41, 42, 45, 47, 50, 52, 65, 76, 78..."
0,Auriculares con Micrófono USB,471,"2, 17, 18, 27, 30, 32, 39, 44, 45, 46, 47, 48,..."


----------------------------
Dejando visible el campo id sólo como un campo representativo


,nombre_producto,unidades_vendidas,ids_venta
2,Escáner de Documentos Portátil,551,2
9,Mouse Ergonómico Inalámbrico,532,5
5,Laptop Corporate 14',504,33
3,Hub USB-C 8 en 1,502,11
7,Memoria RAM 16GB DDR4,492,7
1,Disco Duro Externo 2TB,492,1
6,Licencia Software Antivirus 1 Año,486,4
10,Silla de Escritorio Ergonómica,479,8
11,Teclado Mecánico RGB,475,47
0,Auriculares con Micrófono USB,471,18


## Tear down

In [73]:
# Tear down
conn.close()

## Parte 3 Lógica de negocio

**Parte 3: Lógica de Negocio (Hojas de Cálculo / Excel)**

*Para esta sección, deberás ingeniártelas para extraer/descargar los datos de este entorno (ej. generando archivos CSV desde tus DataFrames o desde la DB) e importarlos a Google Sheets o Excel, puedes utilizar la tecnología que estimes conveniente, lo importante es resolver en Excel.*

1.	**Consolidación**: En una hoja nueva, toma una muestra de las primeras 100 ventas según el dte asociado (a menor folio, el documento es más antiguo). Usa fórmulas de búsqueda (BUSCARV, BUSCARX o INDICE/COINCIDIR) para traer el nombre y apellido del cliente y el nombre_cargo del vendedor asociado a esa venta.

**Comentario**: Usé XLOOKUP

2.	**Tabla Dinámica:** Crea una tabla dinámica que muestre el monto total vendido por cada área de la empresa (nombre_area), filtrando solo el estado "Entregado".

**Comentario**: Usé Power Pivot para crear un modelo de datos y no hacer XLOOKUPs "alterando" la tabla ventas, pero era la opción más sencilla.

3.	**Segmentación:** En tu hoja de ventas, crea la columna "Categoría de Ticket". Usa funciones lógicas para clasificar la venta como "Ticket Alto" (si el total_dte supera el promedio general) o "Ticket Bajo" (si es igual o inferior).


**Comentario**: En este caso sí alteré la tabla ventas agregando la columna porque así lo dice el enunciado.


In [80]:
# Extracción
df_ventas.to_csv('ventas.csv', index=False, encoding='utf-8-sig')
df_clientes.to_csv('clientes.csv', index=False, encoding='utf-8-sig')
df_trabajadores.to_csv('trabajadores.csv', index=False, encoding='utf-8-sig')
df_cargos.to_csv('cargos.csv', index=False, encoding='utf-8-sig')
df_areas.to_csv('areas.csv', index=False, encoding='utf-8-sig')
#df_productos
df_productos = ventas_productos_melted.copy()
df_productos.to_csv('ventas_productos.csv', index=False, encoding='utf-8-sig')

**Opcional (Puntos extra):**

**Presentación al Negocio (Dashboard + Insights)**

1.	**Visualización**: Conecta tus datos exportados a Power BI, Looker o Tableau. Crea un dashboard de una página con:


*   KPI de Ingresos Totales y Ticket Promedio.
*   Gráfico comparativo de rendimiento por categoría/producto.
*   Filtros interactivos.
2.	**Estrategia**: En un documento breve o en celdas de texto al final de este cuaderno, responde:

*   ¿Cuáles fueron tus hallazgos respecto a la fluctuación de ingresos?
*   Propón dos recomendaciones accionables para marketing/ventas.

**Control de versiones (Github):**

1.	**Creación del repositorio:** Entregar los scripts (SQL/Python) subidos a un repositorio propio de GitHub.


# Enlaces

[Parte 3, lógica de negocio Excel](https://1drv.ms/x/c/b2fc1c7e3bed4567/IQBJYLA83XIfSJW5isJ5XqonAQvn2CpzGgAAoFpwUUCmjAg?e=EXc3bs)


[Power BI report](https://app.powerbi.com/view?r=eyJrIjoiMTUyNzYxNWQtZjI2YS00YzNjLWFmYmEtNTk0YjIyNWVlMzU3IiwidCI6IjE2Y2NjYTMyLThjNWQtNGIyNS04Mzk0LWRjNTljYWJlODcwNiJ9)



## ¿Cuáles fueron tus hallazgos respecto a la fluctuación de ingresos? + recomendaciones

Un detalle importante es que la tabla de ventas no tiene un campo fecha de venta y tampoco hay datos de comparación trimestre contra trimestre, por lo que el análisis que podemos hacer se limita a ver las tendencias y patrones dentro del mismo trimestre a evaluar

Al tener un campo de fecha podríamos agrupar las ventas por día, semana, mes y observar la tendencia dentro del mismo trimestre, sin considerar estacionalidad.

Dicho esto voy a concluir en base a:
1. El análisis del notebook
2. Información del Excel
3. Información de Power BI

## Conclusiones del notebook

### Pedidos sin entregar
Hay un 111,4M (18.94%)  de pedidos sin entregar, debemos evaluar si este número es razonable o indica que hay un problema en el área de despacho.

La venta total es de 588M, el promedio mensual 196M (33,33% de la venta total), cuando comparamos el ambos porcentajes podemos ver que la cantidad de pedidos sin entregar es desproporcionada, representaría el 56% de la venta que se realiza en un mes.

Se podría pensar que puede deberse a un exceso de venta ocurrida por un evento especial como el Cyber Day, sin embargo vemos que hay una cantidad importante de productos con id de venta "antiguo" que representarían ventas de meses anteriores.

Si existe un problema en los despachos puede estar afectando las ventas de la empresa.

**Recomendaría revisar los procesos de despacho, aunque esta observación no tiene que ver con marketing y ventas**

### Top 5 clientes según ventas
Hay una serie de clientes que compran muy por encima del promedio. **Recomendaría hacer una campaña diferenciada atendiendo este tipo de clientes incentivandolos a comprar**

### Retención de clientes
La retención de clientes del 24% aparentemente es baja, en todo caso hay que compararla con el histórico y con datos de competidores para entender si está acorde al mercado o no. De cualquier forma 24% parece ser un valor bajo **Recomendaría hacer una campaña sobre este tipo de clientes incentivando la frecuencia de compra, por encima del monto de la compra**


### Promedio de venta por vendedor

El promedio de venta por vendedor es de 32M, hay una serie de vendedores debajo del promedio

Nicole Espinoza	$31.683.368

Francisco Silva	$30.465.402

Carolina Castillo	$30.150.097

Javiera Rojas	$28.122.125

Natalia Cárcamo	$26.223.370

Isabel Martínez	$25.977.742

Juan Donoso	$17.711.014

**Recomendaría apoyar a Juan Donoso y averiguar el por qué de bajo rendimiento y de otros compañeros con bajo promedio, por otro lado incentivar al top 3 vendedores para que se genere una sana intención de mantener ese ritmo de ventas**

### Top 10 productos

Identifiqué 12 productos en total, y acá obtuvimos el top 10, siempre sale la duda: ¿potenciar el marketing de los productos top dado que el mercado los demanda o potenciar el marketing de los productos con poca venta porque pudieran no estar siendo promocionandos?

Tiendo a pensar que hay que optar más por la opción 1 que la opción 2, pero es relativo, si luego de un análisis comparativo se concluye que los productos de baja venta no se venden por problemas de marketing, en esa situación tomar los correctivos.

## Conclusiones del Excel
En Excel no pude observar mayor insight, las tareas allí realizadas están más orientadas a demostrar habilidad.

## Conclusiones de Power BI

### Productos sin entregar
Los productos sin entregar van desde el 20% al 30% según la venta de cada producto, no se observa que haya especial problema en la entrega de un producto en particular.

### Ingresos por categoría de ticket

3/4 de los ingresos se explican por tickets altos por encima del promedio y 1/4 por tickets bajos.

Esto podría ser indicativo de dos claras categorías de clientes, caso que se puede profundizar haciendo una segmentación de clientes por tamaño del ticket, si se confirma la hipótesis y efectivamente los clientes se dividen de esta manera **recomendaría hacer campañas de marketing y ventas diferenciadas, atendiendo las características de estos dos tipos de clientes**


